# Consensus Study Analysis

This notebook imports data collected by Daphne's consensus study, analyzes
scalability and performance metrics of consensus protocols, and visualizes
results.

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
%matplotlib inline

# Enable copy-on-write to follow pandas recommendation.
# see https://pandas.pydata.org/docs/user_guide/copy_on_write.html
pd.options.mode.copy_on_write = True

In [ ]:
# Load collected data from Parquet file.
# Replace '../data.parquet' with the path to your data file.
# As a reminder, please run `go run ./daphne study consensus` to produce this 
# file. Data is loaded in chunks and processed to categorize columns to limit 
# peak memory usage.
data = pd.read_parquet(
    '../../data.parquet',
    columns=['timestamp', 'mark', 'hash', 'rid', 'NumNodes', 'Consensus'],
)

for col in ['mark', 'Consensus']:
    data[col] = data[col].astype('category')

## 2. Time to Finality
The following chart compares the time-to-finality distribution of the various
consensus algorithms. It uses the following definition of time-to-finality:

> The time from submitting the transaction to the first time the transaction got
confirmed in a block by any node.

Future work may refine this definition to more meaningful definitions.

In [ ]:
# Filter only relevant events
tx_submitted = data[data['mark'] == 'TxSubmitted'][['timestamp', 'hash', 'rid']]
tx_finalized = data[data['mark'] == 'TxFinalized'][['timestamp', 'hash', 'rid']]

# Get first TxFinalized per transaction (by hash and rid)
tx_finalized_first = tx_finalized.sort_values('timestamp').drop_duplicates(['hash', 'rid'], keep='first')

# Merge submitted and finalized to compute TTF
ttf = pd.merge(
    tx_submitted,
    tx_finalized_first,
    on=['hash', 'rid'],
    suffixes=('_submitted', '_finalized')
)
ttf['TTF'] = (ttf['timestamp_finalized'] - ttf['timestamp_submitted']) / 1e9  # convert ns to seconds

# Add Consensus info by merging with StudyStarted rows
run_info = data[data['mark'] == 'StudyStarted'][['rid', 'Consensus']]
ttf = pd.merge(ttf, run_info, on='rid', how='left')

# Drop missing Consensus (shouldn't happen, but just in case)
ttf = ttf.dropna(subset=['Consensus'])

# Plot violin plot
plt.figure(figsize=(10, 6))
plt.grid(axis='y', linestyle=':')
sns.violinplot(data=ttf, x='Consensus', y='TTF', inner='quartile')
plt.ylabel('Time to Finality (seconds)')
plt.xlabel('Consensus Algorithm')
plt.title('TTF Distribution by Consensus Algorithm')
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

In [ ]:
# Filter only relevant events
tx_skipped = data[data['mark'] == 'TxSkipped'][['timestamp', 'rid']]

# Add Consensus info by merging with StudyStarted rows
run_info = data[data['mark'] == 'StudyStarted'][['rid', 'Consensus']]
tx_skipped = pd.merge(tx_skipped, run_info, on='rid', how='left')

# Count skipped transactions per Consensus
skipped_counts = tx_skipped.groupby('Consensus', observed=False).size().reset_index(name='SkippedCount')

# Plot skipped transactions
plt.figure(figsize=(10, 6))
sns.barplot(data=skipped_counts, x='Consensus', y='SkippedCount')
plt.ylabel('Number of Skipped Transactions')
plt.xlabel('Consensus Algorithm')
plt.title('Skipped Transactions by Consensus Algorithm')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()